# AI Lab – Experiment 2 & Experiment 3
## Customized CNN for Handwritten Digit Classification (MNIST)

Experiment 3 is the same as Experiment 2 **plus one extra task: display the odd numbers only.**
So this notebook is built in two parts, and one trained model answers both:

| Question part | Experiment 2 | Experiment 3 | Where in this notebook |
|---|---|---|---|
| a) CNN: Conv2D 32, 64 + ReLU + MaxPooling2D (3×3, stride 1) | ✔ | ✔ | Part A, Step 7 |
| b) Flatten → Dense 64 → Dense 10 (Softmax) | ✔ | ✔ | Part A, Step 7 |
| c) Display the numbers | ✔ | ✔ | Part A, Step 3 |
| d) Display the **odd numbers only** | – | ✔ (extra) | **Part B** |
| Training accuracy vs Testing accuracy graph | ✔ (d) | ✔ (e) | Part A, Step 13 |

**How to run:** Google Colab → `Runtime > Run all` (GPU is optional but faster).

# PART A – Experiment 2 (also covers the common parts of Experiment 3)

**Step 1: Import Libraries**

In [ ]:
import numpy as np                       # arrays and maths
import matplotlib.pyplot as plt          # drawing images and graphs
import tensorflow as tf                  # deep learning library

from tensorflow.keras.datasets import mnist                  # MNIST dataset
from tensorflow.keras.models import Sequential               # model: layers one after another
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping         # stops training automatically

from sklearn.model_selection import train_test_split         # splits data

np.random.seed(42)                       # fixed seeds -> repeatable results
tf.random.set_seed(42)

**Step 2: Load MNIST Dataset** (70,000 handwritten digit images, 28×28 pixels, labels 0–9)

In [ ]:
(X_train_original, y_train_original), (X_test_original, y_test_original) = mnist.load_data()

print("Original training data:", X_train_original.shape)
print("Original testing data :", X_test_original.shape)

**Step 2b: Combine Training and Testing Data** (we will make our own split in Step 4)

In [ ]:
X = np.concatenate((X_train_original, X_test_original), axis=0)   # all images together
y = np.concatenate((y_train_original, y_test_original), axis=0)   # all labels together

print("Total images:", X.shape)
print("Total labels:", y.shape)

**Step 3: Display Some MNIST Images** (question part c: display the numbers)

In [ ]:
plt.figure(figsize=(10, 5))

for i in range(10):                              # first 10 images
    plt.subplot(2, 5, i + 1)                     # 2 rows x 5 columns grid
    plt.imshow(X[i], cmap='gray')                # draw the image in gray
    plt.title("Digit: " + str(y[i]))             # its true label
    plt.axis('off')                              # hide the axes

plt.tight_layout()
plt.show()

**Step 4: Train / Test Split** (80 % train, 20 % test; `stratify` keeps all digits balanced)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("Train images:", X_train.shape)
print("Test images :", X_test.shape)

**Step 5: Normalize Images** (pixel values 0–255 → 0–1, so training is faster and more stable)

In [ ]:
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

**Step 6: Add Channel Dimension** (a CNN expects images shaped (height, width, channels); gray image = 1 channel)

In [ ]:
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

print("New training shape:", X_train.shape)
print("New testing shape :", X_test.shape)

**Step 7: Create the CNN Model** (question parts a and b)

- a) Two `Conv2D` layers (32 and 64 filters) each followed by **ReLU** and **MaxPooling2D** (pool size 3×3, stride 1×1)
- b) `Flatten` → `Dense(64)` → `Dense(10, softmax)`

In [ ]:
model = Sequential([

    Input(shape=(28, 28, 1)),                           # input: 28x28 gray image

    # ----- a) First CNN hidden layer: 32 filters + ReLU -----
    Conv2D(32, kernel_size=(3, 3), activation='relu'),

    # First MaxPooling layer: kernel (3,3), stride (1,1)
    MaxPooling2D(pool_size=(3, 3), strides=(1, 1)),

    # ----- a) Second CNN hidden layer: 64 filters + ReLU -----
    Conv2D(64, kernel_size=(3, 3), activation='relu'),

    # Second MaxPooling layer: kernel (3,3), stride (1,1)
    MaxPooling2D(pool_size=(3, 3), strides=(1, 1)),

    # ----- b) Convert the feature maps into a 1D vector -----
    Flatten(),

    # Dense layer of size 64
    Dense(64, activation='relu'),

    # Output layer: 10 classes (digits 0-9) with Softmax
    Dense(10, activation='softmax')
])

**Step 8: Display Model Architecture**

In [ ]:
model.summary()

**Step 9: Compile Model** (`sparse_categorical_crossentropy` because labels are integers 0–9)

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

**Step 10: EarlyStopping Callback** (stops when validation loss stops improving for 3 epochs)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,     # go back to the best epoch
    verbose=1
)

**Step 11: Train Model** (12.5 % of the 80 % training data = 10 % of all data is used for validation)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.125,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

**Step 12: Evaluate: Training Accuracy and Testing Accuracy**

In [ ]:
train_loss, train_accuracy = model.evaluate(X_train, y_train, verbose=0)
test_loss,  test_accuracy  = model.evaluate(X_test,  y_test,  verbose=0)

print(f"Training Accuracy: {train_accuracy*100:.2f} %")
print(f"Testing  Accuracy: {test_accuracy*100:.2f} %")

**Step 13: Training Accuracy vs Testing Accuracy (Graph)**

In [ ]:
plt.figure(figsize=(13, 5))

# Left graph: accuracy during training, with the test accuracy as a dashed line
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.axhline(test_accuracy, linestyle='--', color='red', label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training Accuracy vs Test Accuracy')
plt.legend()
plt.grid(True)

# Right graph: final training accuracy and testing accuracy side by side
plt.subplot(1, 2, 2)
plt.bar(['Training', 'Testing'], [train_accuracy, test_accuracy], color=['tab:blue', 'tab:red'])
plt.ylim(0.9, 1.0)
plt.ylabel('Accuracy')
plt.title('Final Accuracy Comparison')
for i, v in enumerate([train_accuracy, test_accuracy]):
    plt.text(i, v + 0.001, f"{v*100:.2f}%", ha='center')

plt.tight_layout()
plt.show()

**Step 14: Display Predictions** (actual digit vs the digit predicted by the CNN)

In [ ]:
predictions = model.predict(X_test)                      # 10 probabilities for every image
predicted_classes = np.argmax(predictions, axis=1)       # the digit with the highest probability

plt.figure(figsize=(10, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    plt.title("Actual: " + str(y_test[i]) + "\nPredicted: " + str(predicted_classes[i]))
    plt.axis('off')

plt.tight_layout()
plt.show()

# PART B – Experiment 3 (extra task)

Experiment 3 uses the **same CNN, same data and same accuracy graph** as Experiment 2 (Part A above).
The only extra requirement is **d) Display the odd numbers only.**

**Step 15: Display the Odd Numbers Only** (digits 1, 3, 5, 7, 9) – Experiment 3, part d

In [ ]:
# A digit is odd when (digit % 2) == 1. np.where gives the positions of such images.
odd_indices = np.where(y_test % 2 == 1)[0]

plt.figure(figsize=(10, 5))

for i in range(10):                                  # show the first 10 odd digits
    index = odd_indices[i]
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[index].reshape(28, 28), cmap='gray')
    plt.title("Odd digit: " + str(y_test[index]))
    plt.axis('off')

plt.suptitle("Odd Numbers Only (from MNIST)")
plt.tight_layout()
plt.show()

**Step 16 (extra): Odd Numbers with the CNN's Predictions** – checks that the model recognises odd digits correctly

In [ ]:
plt.figure(figsize=(10, 5))

for i in range(10):
    index = odd_indices[i]
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[index].reshape(28, 28), cmap='gray')
    plt.title("Actual: " + str(y_test[index]) + "\nPredicted: " + str(predicted_classes[index]))
    plt.axis('off')

plt.tight_layout()
plt.show()

# Accuracy on odd digits only
odd_accuracy = np.mean(predicted_classes[odd_indices] == y_test[odd_indices])
print(f"Accuracy on odd digits only: {odd_accuracy*100:.2f} %")

## Short answers for the viva
- **Conv2D:** slides small filters (3×3) over the image to detect features such as edges and curves. 32 filters in layer 1, 64 filters in layer 2.
- **ReLU:** keeps positive values and sets negatives to 0; adds non-linearity so the network can learn complex patterns.
- **MaxPooling2D (3×3, stride 1):** keeps the strongest value in each 3×3 window, reducing noise and size.
- **Flatten:** turns the 2D feature maps into one long 1D vector for the Dense layers.
- **Softmax (10 neurons):** gives the probability of each digit 0–9; the largest is the prediction.
- **Why normalize (÷255)?** Small input values (0–1) make training faster and more stable.
- **EarlyStopping:** stops training when validation loss stops improving, which prevents overfitting.
- **Odd numbers:** a digit is odd if `digit % 2 == 1` (1, 3, 5, 7, 9).
- **Training vs testing accuracy:** training accuracy is measured on data the model learned from; testing accuracy on unseen data. A big gap means overfitting.